# Gated Consensus Ensemble — Option C: Per-method, per-cluster thresholds

Instead of one threshold per method (Option A), use a **2D threshold dict**:
method x cluster. This gives full control — you can set tight thresholds
for clusters where a method is bad and looser thresholds where it's good.

Values were chosen from the threshold sweep analysis.

- `None` = method silenced for that cluster
- `(op, value)` = threshold to apply

**LT data only.**

## 1. Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os, pickle

%matplotlib inline

results = pickle.load(open('metrics_results.pkl', 'rb'))
constraint_patternsLT = results['constraint_patternsLT']
all_methodsLT = results['all_methodsLT']
method_names = list(all_methodsLT.keys())

cluster_names = list(constraint_patternsLT.keys())

print('Methods:', method_names)
print('Clusters:', cluster_names)
print()
for cn in cluster_names:
    n_genes = len(list(all_methodsLT.values())[0][0][cn]['constraint'])
    print(f'  {cn}: {n_genes:,} genes')

## 2. Per-method, per-cluster thresholds

The key idea: instead of one global threshold per method, each cell in the
method x cluster grid can have its own threshold or be silenced entirely.

- `None` = silenced (method excluded for this cluster)
- `(op, value)` = threshold to apply

In [ ]:
# ============================================================
# OPTION C: 2D THRESHOLDS — method x cluster
#
# None = silenced (method excluded for this cluster)
# (op, value) = threshold to apply
#
# Values from threshold sweep plots in plots/threshold_sweep/.
# ============================================================

THRESHOLDS_2D = {
    'Pearson': {
        'clusterOneLT':   ('>', 0.995),   # tight — Pearson is strong here
        'clusterTwoLT':   ('>', 0.99),    # slightly looser — fewer genes at 0.995
        'clusterThreeLT': None,           # silenced — genes don't overlay pattern
        'clusterFourLT':  None,           # silenced — delta function defeats Pearson
    },
    'DTW': {
        'clusterOneLT':   ('<', 0.08),    # good shape match at this threshold
        'clusterTwoLT':   ('<', 0.08),    # same — works well on monotonic patterns
        'clusterThreeLT': None,           # silenced — genes at wrong magnitude
        'clusterFourLT':  None,           # silenced — spike pattern defeats DTW
    },
    'Cosine': {
        'clusterOneLT':   None,           # silenced everywhere — cosine conflates
        'clusterTwoLT':   None,           # magnitude with shape, not useful here
        'clusterThreeLT': None,
        'clusterFourLT':  None,
    },
    'Frechet': {
        'clusterOneLT':   ('<', 0.03),    # tight geometric match
        'clusterTwoLT':   ('<', 0.03),    # same threshold works
        'clusterThreeLT': None,           # silenced — wrong magnitude
        'clusterFourLT':  None,           # silenced — spike
    },
    'MSE': {
        'clusterOneLT':   ('<', 0.001),   # green in sweep — 25 genes
        'clusterTwoLT':   ('<', 0.001),   # green — 3 genes
        'clusterThreeLT': None,           # silenced — genes at wrong magnitude
        'clusterFourLT':  None,           # silenced — spike
    },
    'sMAPE': {
        'clusterOneLT':   None,           # silenced — 0 genes qualify
        'clusterTwoLT':   ('<', 0.10),    # green — 53 genes
        'clusterThreeLT': ('<', 0.10),    # green — 80 genes, the STAR method here
        'clusterFourLT':  None,           # silenced — spike
    },
}

TOP_N = 20

# Print the 2D threshold grid
print('Per-method, per-cluster thresholds:')
print(f'{"":>10}', end='')
for cn in cluster_names:
    short = cn.replace('cluster', 'c').replace('LT', '')
    print(f'{short:>16}', end='')
print()
print('-' * (10 + 16 * len(cluster_names)))
for mname in method_names:
    print(f'{mname:>10}', end='')
    for cn in cluster_names:
        thresh = THRESHOLDS_2D[mname][cn]
        if thresh is None:
            cell = 'SILENCED'
        else:
            op, val = thresh
            cell = f'{op}{val}'
        print(f'{cell:>16}', end='')
    print()

## 3. Qualifying gene function (2D lookup)

In [ ]:
def get_qualifying_genes_2d(methods_dict, cluster_name, pattern_name, thresholds_2d):
    """For each method, look up its cluster-specific threshold in the 2D dict.
    
    Returns dict: method_name -> Series of scores (only qualifying genes),
    or method_name is absent if silenced (None) for this cluster.
    """
    qualifying = {}
    for mname, (scores, ascending) in methods_dict.items():
        if mname not in thresholds_2d:
            continue
        thresh = thresholds_2d[mname].get(cluster_name)
        if thresh is None:
            # Method silenced for this cluster
            continue
        op, val = thresh
        s = scores[cluster_name][pattern_name]
        if op == '>':
            mask = s > val
        elif op == '<':
            mask = s < val
        elif op == '>=':
            mask = s >= val
        elif op == '<=':
            mask = s <= val
        else:
            raise ValueError(f'Unknown operator: {op}')
        qualifying[mname] = s[mask].sort_values(ascending=ascending)
    return qualifying

## 4. Qualifying gene counts — method x cluster table

In [ ]:
print('QUALIFYING GENE COUNTS (LT DATA) — Option C 2D thresholds')
print('=' * 80)

# Build a counts table for display
counts_data = {}
for cn in cluster_names:
    q = get_qualifying_genes_2d(all_methodsLT, cn, 'constraint', THRESHOLDS_2D)
    col = {}
    for mname in method_names:
        thresh = THRESHOLDS_2D[mname].get(cn)
        if thresh is None:
            col[mname] = 'SILENCED'
        else:
            n = len(q.get(mname, []))
            op, val = thresh
            col[mname] = f'{n:,} ({op}{val})'
    counts_data[cn] = col

counts_df = pd.DataFrame(counts_data)
print()
print(counts_df.to_string())

# Also print numeric counts for quick scanning
print()
print('Numeric counts only:')
for cn in cluster_names:
    q = get_qualifying_genes_2d(all_methodsLT, cn, 'constraint', THRESHOLDS_2D)
    active = [m for m in method_names if m in q]
    silenced = [m for m in method_names if m not in q]
    print(f'\n  {cn}:')
    print(f'    Active:   {len(active)} methods — {", ".join(active) if active else "none"}')
    if silenced:
        print(f'    Silenced: {", ".join(silenced)}')
    for mname in active:
        print(f'      {mname:>8}: {len(q[mname]):>6,} genes')

## 5. Plot qualifying genes per method per cluster

Side-by-side subplots, one per active method. Silenced methods are skipped.

In [ ]:
x = [0, 3, 6, 9]

for cluster_name in cluster_names:
    gene_file = f'analysis data/gene_countsLT/{cluster_name}_annotated.csv'
    gene_df = pd.read_csv(gene_file, index_col=0)
    gene_df.columns = x
    pattern_vals = constraint_patternsLT[cluster_name]['constraint']
    
    q = get_qualifying_genes_2d(all_methodsLT, cluster_name, 'constraint', THRESHOLDS_2D)
    active = [m for m in method_names if m in q]
    silenced = [m for m in method_names if m not in q]
    
    if len(active) == 0:
        print(f'{cluster_name}: ALL METHODS SILENCED — no plot generated')
        continue
    
    n_active = len(active)
    fig, axes = plt.subplots(1, n_active, figsize=(5 * n_active, 4.5), squeeze=False)
    
    for i, mname in enumerate(active):
        ax = axes[0, i]
        genes = q[mname].index.tolist()
        n_genes = len(genes)
        plotted = 0
        for gene in genes[:50]:  # cap at 50 for readability
            if gene in gene_df.index:
                ax.plot(x, gene_df.loc[gene], color='steelblue',
                        alpha=0.3, linewidth=0.8)
                plotted += 1
        ax.plot(x, pattern_vals, color='crimson', linewidth=2.5, linestyle='--')
        
        thresh = THRESHOLDS_2D[mname][cluster_name]
        op, val = thresh
        ax.set_title(f'{mname} ({op}{val})\n{n_genes:,} genes qualify', fontsize=10)
        ax.set_xlabel('Timepoint')
        ax.set_xticks(x)
        if i == 0:
            ax.set_ylabel('Gene counts')
        ax.grid(alpha=0.2)
    
    sil_str = ', '.join(silenced) if silenced else 'none'
    fig.suptitle(f'{cluster_name} (LT) — silenced: {sil_str}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Gated consensus: cluster-specific voting

A gene gets a vote from a method **only if**:
1. The method is not silenced for this cluster (`None` in the 2D dict)
2. The gene's score passes that method's cluster-specific threshold

This means each cluster can have a different set of active methods with
different strictness levels.

In [ ]:
def gated_consensus_2d(methods_dict, cluster_name, pattern_name,
                        thresholds_2d, top_n=20):
    """Score-gated ensemble using 2D (method x cluster) thresholds.
    
    A gene gets a vote from a method only if:
      - the method is not silenced for this cluster
      - the gene passes that method's cluster-specific threshold
    
    Returns (result_df, active_methods, silenced_methods).
    result_df has columns: votes, max_possible, mean_rank, qualified_in.
    """
    q = get_qualifying_genes_2d(methods_dict, cluster_name, pattern_name,
                                 thresholds_2d)
    active = [m for m in method_names if m in q]
    silenced = [m for m in method_names if m not in q]
    
    if len(active) == 0:
        return pd.DataFrame(), active, silenced
    
    # Build rank df for active methods
    rank_df = pd.DataFrame()
    for mname in active:
        scores, ascending = methods_dict[mname]
        s = scores[cluster_name][pattern_name]
        rank_df[mname] = s.rank(ascending=ascending, method='min')
    
    # A gene gets a vote from a method only if it's in that method's qualifying set
    vote_df = pd.DataFrame(index=rank_df.index)
    for mname in active:
        qualifying_genes = set(q[mname].index)
        vote_df[mname] = rank_df.index.isin(qualifying_genes).astype(int)
    
    votes = vote_df.sum(axis=1)
    mean_rank = rank_df[active].mean(axis=1)
    
    # Track which methods each gene qualified in
    def get_qualified_methods(gene):
        return ', '.join(m for m in active if vote_df.loc[gene, m] == 1)
    
    result = pd.DataFrame({
        'votes': votes,
        'max_possible': len(active),
        'mean_rank': mean_rank,
    })
    # Only keep genes with at least 1 vote
    result = result[result['votes'] > 0]
    result = result.sort_values(['votes', 'mean_rank'], ascending=[False, True])
    result = result.head(top_n)
    
    # Add qualified_in column
    result['qualified_in'] = [get_qualified_methods(g) for g in result.index]
    
    return result, active, silenced

## 7. Results table per cluster

In [ ]:
print('GATED CONSENSUS ENSEMBLE — Option C (2D thresholds, LT data)')
print('=' * 90)

ensemble_results = {}  # store for plotting

for cluster_name in cluster_names:
    result, active, silenced = gated_consensus_2d(
        all_methodsLT, cluster_name, 'constraint', THRESHOLDS_2D, top_n=TOP_N
    )
    ensemble_results[cluster_name] = (result, active, silenced)
    
    print(f'\n{"=" * 90}')
    print(f'{cluster_name}')
    print(f'  Active:   {len(active)}/{len(method_names)} — {", ".join(active) if active else "none"}')
    if silenced:
        print(f'  Silenced: {", ".join(silenced)}')
    
    if result.empty:
        print('  NO QUALIFYING GENES — loosen thresholds or enable more methods')
        continue
    
    max_v = len(active)
    print(f'  Genes with {max_v}/{max_v} votes: {(result["votes"] == max_v).sum()}')
    print(f'  Genes with any votes:  {len(result)}')
    print()
    print(f'  {"Rank":<6}{"Gene":<22}{"Votes":>8}  {"Mean Rank":>10}  Qualified in')
    print(f'  {"-" * 80}')
    for i, (gene, row) in enumerate(result.iterrows(), 1):
        v = int(row['votes'])
        mr = row['mean_rank']
        qm = row['qualified_in']
        print(f'  {i:<6}{gene:<22}{v:>5}/{max_v}  {mr:>10.1f}  {qm}')

## 8. Ensemble plots with opacity

Opacity encodes consensus strength: darker lines = more methods agree.

In [ ]:
x = [0, 3, 6, 9]

for cluster_name in cluster_names:
    result, active, silenced = ensemble_results[cluster_name]
    if result.empty:
        print(f'{cluster_name}: no qualifying genes — skipping plot')
        continue
    
    gene_file = f'analysis data/gene_countsLT/{cluster_name}_annotated.csv'
    gene_df = pd.read_csv(gene_file, index_col=0)
    gene_df.columns = x
    pattern_vals = constraint_patternsLT[cluster_name]['constraint']
    
    max_votes = len(active)
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for gene in result.index:
        if gene in gene_df.index:
            v = result.loc[gene, 'votes']
            alpha = 0.2 + 0.6 * (v / max_votes)
            ax.plot(x, gene_df.loc[gene], color='steelblue',
                    alpha=alpha, linewidth=1)
    
    ax.plot(x, pattern_vals, color='crimson', linewidth=2.5, linestyle='--')
    
    sil_str = ', '.join(silenced) if silenced else 'none'
    act_str = ', '.join(active)
    n_genes = len(result)
    title = (f'Gated Consensus (Option C) | {cluster_name} (LT) | '
             f'Top {n_genes} genes\n'
             f'Active: {act_str} | Silenced: {sil_str}')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Timepoint')
    ax.set_ylabel('Gene counts')
    ax.set_xticks(x)
    
    legend_elements = [
        Line2D([0], [0], color='crimson', linewidth=2.5, linestyle='--',
               label='Constraint pattern'),
        Line2D([0], [0], color='steelblue', alpha=0.8,
               label=f'High consensus (>={max_votes-1}/{max_votes} votes)'),
        Line2D([0], [0], color='steelblue', alpha=0.3,
               label='Low consensus (1 vote)'),
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    ax.grid(alpha=0.15)
    plt.tight_layout()
    plt.show()